# Naija ASR Benchmark — Public Results

**Kaggle Benchmark**: Nigerian English & Pidgin ASR Error Classification

This notebook loads the corrected benchmark metrics and visualizes how standard WER
overestimates real ASR errors for dialect-rich speech.

## Key Metrics
| Metric | Meaning |
|--------|---------|
| **raw_wer** | Standard WER — penalizes every substitution/deletion/insertion |
| **faithful_wer** | WER excluding faithful dialect variants (e.g., "people" ↔ "pipo") |
| **dialect_penalty** | raw_wer — faithful_wer — the unfair penalty for dialect transcription |
| **hallucination_rate** | % of word errors that are genuine ASR failures |
| **normalization_overcorrection_rate** | % of hallucinations where model "corrected" dialect to standard English |


## 1. Setup — Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'jiwer'], check=True)
print('Done.')


## 2. Load Data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)

DATA_DIR = Path("/kaggle/input/naija-asr-benchmark-results")
if not DATA_DIR.is_dir():
    DATA_DIR = Path("../../data/kaggle_benchmark/analysis")  # local fallback

df = pd.read_csv(DATA_DIR / "benchmark_eval_results.csv")
df['value'] = pd.to_numeric(df['value'], errors='coerce')

models_order = ["Whisper large-v3", "Whisper small", "Accent Labs fine-tuned"]
dialects_order = ["hausa", "igbo", "multilingual", "pidgin", "yoruba"]

def _model_dict(mid):
    sub = df[(df['section'].isin(['summary','per_dialect'])) & (df['model_id'] == mid)]
    d = {}
    for _, r in sub[(sub['section']=='summary') & (sub['accent_background']=='overall')].iterrows():
        d[r['metric']] = float(r['value'])
    for k in ['raw_wer','faithful_wer','hallucination_rate']:
        if k in d: d[k] = d[k] / 100.0
    if 'dialect_penalty' in d: d['dialect_penalty'] = d['dialect_penalty'] / 100.0
    dia = {}
    pen = {}
    for _, r in sub[sub['section']=='per_dialect'].iterrows():
        if r['metric'] == 'raw_wer': dia[r['accent_background']] = float(r['value']) / 100.0
        if r['metric'] == 'dialect_penalty': pen[r['accent_background']] = float(r['value']) / 100.0
    d['dialects'] = dia
    d['dialect_penalties'] = pen
    return d

models_summary = {m: _model_dict(m) for m in models_order}
print(f'Sections: {df["section"].unique()}')


## 3. Per-Dialect WER — 3 Models Side-by-Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (model_id, model_data) in enumerate(models_summary.items()):
    ax = axes[idx] if idx < 2 else axes[1]
    dialects = list(model_data['dialects'].keys())
    wers = [model_data['dialects'][d] for d in dialects]
    penalties = [model_data['dialect_penalties'][d] for d in dialects]
    x = np.arange(len(dialects))
    width = 0.35
    ax.bar(x - width/2, wers, width, label='Raw WER', color='#4C72B0')
    ax.bar(x + width/2, penalties, width, label='Dialect Penalty', color='#DD8452')
    ax.set_xlabel('Dialect')
    ax.set_ylabel('WER')
    ax.set_title(f"{model_id}\nRaw WER: {model_data['raw_wer']:.2%}")
    ax.set_xticks(x)
    ax.set_xticklabels([d.title() for d in dialects], rotation=25)
    ax.legend()

plt.suptitle('Dialect-Aware WER by Model and Dialect', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Combined view
fig, ax = plt.subplots(figsize=(14, 6))
all_dialects = sorted(set(d for m in models_summary.values() for d in m['dialects']))
x = np.arange(len(all_dialects))
n_models = len(models_order)
width = 0.8 / n_models
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, mid in enumerate(models_order):
    wers = [models_summary[mid]['dialects'].get(d, 0) for d in all_dialects]
    offset = (i - n_models/2 + 0.5) * width
    ax.bar(x + offset, wers, width, label=mid, color=colors[i % len(colors)])
ax.set_xlabel('Dialect')
ax.set_ylabel('Raw WER')
ax.set_title('Raw WER Comparison Across Models')
ax.set_xticks(x)
ax.set_xticklabels([d.title() for d in all_dialects], rotation=25)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Dialect Penalty Analysis

In [ ]:
penalty_data = []
for mid, m in models_summary.items():
    for d in m['dialects']:
        penalty_data.append({'model': mid, 'dialect': d.title(),
            'dialect_penalty': m['dialect_penalties'][d],
            'raw_wer': m['dialects'][d],
            'faithful_wer': m['dialects'][d] - m['dialect_penalties'][d]})
penalty_df = pd.DataFrame(penalty_data)
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=penalty_df, x='dialect', y='dialect_penalty', hue='model', ax=ax)
ax.set_title('Dialect Penalty by Dialect and Model')
plt.tight_layout()
plt.show()


## 5. Hallucination Rate by Model and Dialect

In [ ]:
hall_data = []
pdf = df[df['section'] == 'per_dialect']
for mid in models_order:
    for d in dialects_order:
        row = pdf[(pdf['model_id']==mid) & (pdf['accent_background']==d)]
        errs = row[row['metric']=='total_errors']['value'].iloc[0] if len(row[row['metric']=='total_errors']) else 0
        halls = row[row['metric']=='hallucinations']['value'].iloc[0] if len(row[row['metric']=='hallucinations']) else 0
        faiths = row[row['metric']=='faithful_variants']['value'].iloc[0] if len(row[row['metric']=='faithful_variants']) else 0
        hall_rate = halls / errs if errs else 1.0
        hall_data.append({'model':mid,'dialect':d.title(),'hallucination_rate':hall_rate,
            'total_errors':int(errs),'hallucinations':int(halls),'faithful_variants':int(faiths)})
hall_df = pd.DataFrame(hall_data)
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=hall_df, x='dialect', y='hallucination_rate', hue='model', ax=ax)
ax.set_title('Hallucination Rate (among errors)')
ax.set_ylabel('Rate')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 6. Normalization Overcorrection Rate

In [ ]:
norm_data = []
for mid in models_order:
    sub = df[(df['section']=='per_dialect') & (df['model_id']==mid) & (df['metric']=='overcorrection_rate')]
    for _, r in sub.iterrows():
        norm_data.append({'model':mid,'dialect':r['accent_background'].title(),'ovcorr_rate':r['value']})
norm_df = pd.DataFrame(norm_data)
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=norm_df, x='dialect', y='ovcorr_rate', hue='model', ax=ax)
ax.set_title('Normalization Overcorrection Rate')
ax.set_ylabel('Overcorrection Rate (%)')
plt.tight_layout()
plt.show()


## 7. Summary Table

In [ ]:
summary_rows = []
for mid in models_order:
    m = models_summary[mid]
    summary_rows.append({'Model':mid,
        'Raw WER': f"{m['raw_wer']:.2%}",
        'Faithful WER': f"{m['faithful_wer']:.2%}",
        'Dialect Penalty': f"{m['dialect_penalty']*100:.2f}",
        'Hallucination Rate': f"{m['hallucination_rate']:.1%}",
        'Overcorrection': f"{df[(df['section']=='summary')&(df['model_id']==mid)&(df['metric']=='overcorrection_pct')]['value'].iloc[0]:.1f}%"})
display(pd.DataFrame(summary_rows).set_index('Model'))


## Held-Out WER (160 Non-Overlapping Samples)

340 of 500 test sample IDs (68%) overlap the fine-tuned model's training data.
The following numbers are on the 160 clean (non-overlapping) samples only:


In [ ]:
heldout = df[df['section']=='heldout'].pivot_table(
    index='model_id', columns='metric', values='value', aggfunc='first')
display(heldout)


## Punctuation Artifact

The original WER run didn't strip punctuation before jiwer alignment — trailing periods
and commas on Whisper's default output inflated baseline WER ("people.," counted as an
error against "people"). After fixing this, the corrected numbers are reported above.
Raw originals are kept in `results_original.json`.


## Cause Distribution (% of hallucinations)

In [ ]:
cause = df[df['section']=='cause_distribution']
cause_pivot = cause[cause['metric'].str.endswith('_pct')].pivot_table(
    index='model_id', columns='metric', values='value', aggfunc='first')
display(cause_pivot)

fig, ax = plt.subplots(figsize=(10, 6))
cause_pct = cause[cause['metric'].isin(['phonetic_pct','dialect_norm_pct','noise_pct'])]
cp = cause_pct.pivot_table(index='model_id', columns='metric', values='value', aggfunc='first')
cp.plot(kind='bar', stacked=True, ax=ax, color=['#4C72B0','#DD8452','#55A868'])
ax.set_ylabel('% of hallucinations')
ax.set_title('Cause Distribution by Model')
ax.legend(['Phonetic','Dialect Normalization','Noise'])
plt.tight_layout()
plt.show()


## Methodology

### Error Taxonomy
- **faithful_dialect**: Model transcribed the dialect correctly, but reference uses standard English orthography.
- **hallucination (phonetic)**: Genuine ASR failure — output doesn't match acoustic signal. Also the default
  catch-all for substitutions not matching a known dialect pair.
- **hallucination (dialect_normalization)**: Over-correction — model "fixes" dialect to standard English.
- **hallucination (noise)**: Inserted word with no counterpart in the reference transcript.

### WER Computation
```python
import re, jiwer
text = re.sub(r"[^\w\s]", "", text.lower())
jiwer.process_words(ref_clean, hyp_clean)
```
Punctuation stripped and lowercased before alignment.

### Dialect Classifier
38 FAITHFUL_DIALECT_PAIRS + 82 DIALECT_NORMALIZATION_PAIRS.
Unmatched substitutions default to `phonetic`. Insertions → `noise`.

### Benchmark Design
- 500 samples stratified across 5 ethnolinguistic backgrounds (Hausa, Igbo, Multilingual, Pidgin, Yoruba)
- Max 15 samples per speaker
- 3 Whisper models: large-v3 (zero-shot), small (zero-shot), fine-tuned
- 340 of 500 test sample IDs overlap fine-tuning data; 160 held-out samples reported separately

### Data
- `benchmark_eval_results.csv`: Summary metrics, per-dialect breakdown, cause distribution, held-out WER
- Full outputs: `results.json`, `metrics_per_dialect.csv`, `word_alignment.csv` (see GitHub repo)
